In [21]:
import jax.numpy as jnp
from jax.typing import ArrayLike
from jax import Array

from typing import Optional, Callable

from chromatix import Field, ScalarField, VectorField
import jax
from chromatix.utils import l2_sq_norm

In [31]:
def point_source(
    shape: tuple[int, int],
    dx: ArrayLike,
    spectrum: ArrayLike,
    spectral_density: ArrayLike,
    z: ArrayLike,
    n: float,
    power: float = 1.0,
    amplitude: ArrayLike = 1.0,
    pupil: Optional[Callable[[Field], Field]] = None,
) -> Field:
    """
    Generates field due to point source a distance ``z`` away.

    Can also be given ``pupil``.

    Args:
        shape: The shape (height and width) of the ``Field`` to be created.
        dx: The spacing of the samples of the ``Field``.
        spectrum: The wavelengths included in the ``Field`` to be created.
        spectral_density: The weights of each wavelength in the ``Field`` to
            be created.
        z: The distance of the point source.
        n: Refractive index.
        power: The total power that the result should be normalized to,
            defaults to 1.0.
        amplitude: The amplitude of the electric field. For ``ScalarField`` this
            doesnt do anything, but it is required for ``VectorField`` to set
            the polarization.
        pupil: If provided, will be called on the field to apply a pupil.
        scalar: Whether the result should be ``ScalarField`` (if True) or
            ``VectorField`` (if False). Defaults to True.
    """
    @jax.vmap
    def forward(field, z):
        # TODO: Check if negative z is correct
        l_sq = field.spectrum * z / n 
        phase = jnp.pi / l_sq * l2_sq_norm(field.grid)
        u = -1j * amplitude / l_sq * jnp.exp(1j * phase)
        return field.replace(u=u)

    create = ScalarField.create if scalar else VectorField.create
    field = create(dx, spectrum, spectral_density, shape=shape)
    field = forward(field, jnp.atleast_1d(z))
    
    if pupil is not None:
        field = pupil(field)
        
    return field * jnp.sqrt(power / field.power)


In [32]:
field = point_source((512, 512), dx=0.1, spectrum=0.532, spectral_density=1.0, z=100, n=1.33)

In [33]:
field.u.shape

(1, 512, 512, 1, 1)

In [35]:
field.dx.shape

(2, 1, 1, 1, 1, 1)

In [ ]:
field.